In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pylab as plt
import datetime
import scipy.stats
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.dates as mdates
import matplotlib


In [ ]:
def decimal_year_to_datetime(decimal_year):
    year = decimal_year.astype(int)
    decimal_part = decimal_year - year
    start_of_year = pd.to_datetime([f'{y}-01-01' for y in year])
    seconds_in_year = (pd.to_datetime([f'{y+1}-01-01' for y in year]) - start_of_year).total_seconds()
    seconds = decimal_part * seconds_in_year
    return start_of_year + pd.to_timedelta(seconds, unit='s')

In [ ]:
decimal_year_to_datetime(np.array([1961.1]))

In [ ]:
window_size = 3

df = pd.read_excel('merged_corrected.xlsx')
#df[['d13c_ma','d18o_ma']] = df[['d13c','d18o']].rolling(window_size, center=True).mean()
df_sorted = df.sort_values("age_absolute")
df_sorted[['d13c_ma','d18o_ma']] = df_sorted[['d13c','d18o']].rolling(window_size, center=True).mean()
df_sorted

In [ ]:
fig, axes = plt.subplots(2,1, figsize=(8, 6), sharex=True, gridspec_kw={'hspace':0})

ax = axes[0]
ax.plot(df_sorted['age_absolute'], df_sorted['d13c'], ".", color='lightgreen')
ax.plot(df_sorted['age_absolute'],df_sorted['d13c_ma'], color='green')
ax.grid()

ax = axes[1]
ax.plot(df_sorted['age_absolute'],df_sorted['d18o'], ".-")
ax.plot(df_sorted['age_absolute'],df_sorted['d18o_ma'])
ax.grid()

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=df_sorted['age_absolute'], y=df_sorted['d13c'],  mode="markers", marker_color="lightgreen",  name='d13C'))
fig.add_trace(go.Scatter(x=df_sorted['age_absolute'], y=df_sorted['d13c_ma'], mode="lines", line_color='green', name='average'))
fig.show()
#fig.write_html('d13C.html')

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=df_sorted['age_absolute'], y=df_sorted['d18o'],  mode="markers", marker_color="lightblue",  name='d18O'))
fig.add_trace(go.Scatter(x=df_sorted['age_absolute'], y=df_sorted['d18o_ma'], mode="lines", line_color='blue', name='average'))
fig.show()
#fig.write_html('d18O.html')

In [ ]:
enso = pd.read_csv("../ENSO_Time_Series.csv", parse_dates=[0])
#enso = enso.set_index("Date")
enso['day_of_year'] = enso['Date'].dt.dayofyear
enso['is_leap'] = enso['Date'].dt.is_leap_year
enso['total_days'] = np.where(enso['is_leap'], 366, 365)
enso['decimal_year'] = enso['Date'].dt.year + (enso['day_of_year'] - 1) / enso['total_days']
enso

In [ ]:
enso_phase = pd.read_csv('../ENSO_Phase_Classification.csv')
enso_phase.columns=['Year','Phase']
enso_phase['Date'] = decimal_year_to_datetime(enso_phase['Year'])
#enso_phase.set_index('Date', inplace=True)
enso_phase['Phase'] = enso_phase['Phase'].str.replace('Warm Phase', 'W').str.replace('Cold Phase', 'C').str.replace('Neutral Phase','N')
enso_phase

In [ ]:
#fig = go.Figure()
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=df_sorted['age_absolute'], y=df_sorted['d18o'],  mode="markers",
                         marker_color="lightblue",  name='d18O'),secondary_y=False)
fig.add_trace(go.Scatter(x=df_sorted['age_absolute'], y=df_sorted['d18o_ma'], mode="lines",
                         line_color='blue', name='average'), secondary_y=False)

fig.add_trace(go.Scatter(x=enso['decimal_year'], y=enso['ENSO'], mode="lines", line_color='red', name='ENSO'), secondary_y=True)

fig.add_vline(x=1935.25, line_width=2, line_dash="dash", line_color="orange", opacity=1, showlegend=True,name='Gizo EQ 1935 Mar')
fig.add_vline(x=1966, line_width=2, line_dash="dash", line_color="violet", opacity=1, showlegend=True,name='Kavachi submarine volcano 1966')
fig.add_vline(x=2007, line_width=2, line_dash="dash", line_color="purple", opacity=1, showlegend=True,name='Gizo 2007 EQ M8.1')
fig.add_vline(x=1977.3, line_width=2, line_dash="dash", line_color="gold", opacity=1, showlegend=True,name='Guadalcanal 1977 EQ M9+')


# Add figure title
fig.update_layout(title_text="d18O and ENSO", template='simple_white')

# Set x-axis title
fig.update_xaxes(title_text="Year")

# Set y-axes titles
fig.update_yaxes(title_text= '$\delta O ^{18}$', secondary_y=False)
fig.update_yaxes(title_text="ENSO Index", secondary_y=True)

fig.show()
fig.write_html('d18O_ENSO.html')


In [ ]:
#fig = go.Figure()
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=df_sorted['age_absolute'], y=df_sorted['d13c'],  mode="markers", marker_color="lightgreen",  name='d18O'),secondary_y=False)
fig.add_trace(go.Scatter(x=df_sorted['age_absolute'], y=df_sorted['d13c_ma'], mode="lines", line_color='green', name='average'), secondary_y=False)

fig.add_trace(go.Scatter(x=enso['decimal_year'], y=enso['ENSO'], mode="lines", line_color='red', name='ENSO'), secondary_y=True)

fig.add_vline(x=1935.25, line_width=2, line_dash="dash", line_color="orange", opacity=1, showlegend=True,name='Gizo EQ 1935 Mar')
fig.add_vline(x=1966, line_width=2, line_dash="dash", line_color="violet", opacity=1, showlegend=True,name='Kavachi submarine volcano 1966')
fig.add_vline(x=2007, line_width=2, line_dash="dash", line_color="purple", opacity=1, showlegend=True,name='Gizo 2007 EQ M8.1')
fig.add_vline(x=1977.3, line_width=2, line_dash="dash", line_color="gold", opacity=1, showlegend=True,name='Guadalcanal 1977 EQ M9+')





# Add figure title
fig.update_layout(title_text="d13C and ENSO", template='simple_white')
# Set x-axis title
fig.update_xaxes(title_text="Year")

# Set y-axes titles
fig.update_yaxes(title_text= '$\delta C ^{13}$', secondary_y=False)
fig.update_yaxes(title_text="ENSO Index", secondary_y=True)

fig.show()
fig.write_html('d13C_ENSO.html')


In [ ]:
phase_color = {'W':'red', 'C':'blue'}
enso_phase_selected = enso_phase[(enso_phase['Year']>1925)&(enso_phase['Year']<2008)&(enso_phase['Phase'] != 'N')]

#fig = go.Figure()
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=df_sorted['age_absolute'], y=df_sorted['d13c'],  mode="markers", marker_color="lightgreen",  name='d18O'),secondary_y=False)
fig.add_trace(go.Scatter(x=df_sorted['age_absolute'], y=df_sorted['d13c_ma'], mode="lines", line_color='green', name='average'), secondary_y=False)

fig.add_trace(go.Scatter(x=enso['decimal_year'], y=enso['ENSO'], mode="lines", line_color='gray', name='ENSO'), secondary_y=True)

fig.add_vline(x=1935.25, line_width=2, line_dash="dash", line_color="orange", opacity=1, showlegend=True,name='Gizo EQ 1935 Mar')
fig.add_vline(x=1966, line_width=2, line_dash="dash", line_color="violet", opacity=1, showlegend=True,name='Kavachi submarine volcano 1966')
fig.add_vline(x=2007, line_width=2, line_dash="dash", line_color="purple", opacity=1, showlegend=True,name='Gizo 2007 EQ M8.1')
fig.add_vline(x=1977.3, line_width=2, line_dash="dash", line_color="gold", opacity=1, showlegend=True,name='Guadalcanal 1977 EQ M9+')

for i, index in enumerate(enso_phase_selected.index):
    
    df_iter = enso_phase.loc[index]
    df_iter_next = enso_phase.loc[index+1]
    fig.add_vrect(x0=df_iter['Year'], x1=df_iter_next['Year'], fillcolor=phase_color[df_iter['Phase']], opacity=0.1)




# Add figure title
fig.update_layout(title_text="d13C and ENSO", template='simple_white')
# Set x-axis title
fig.update_xaxes(title_text="Year")

# Set y-axes titles
fig.update_yaxes(title_text= '$\delta C ^{13}$', secondary_y=False)
fig.update_yaxes(title_text="ENSO Index", secondary_y=True)

fig.show()
fig.write_html('d13C_ENSO.html')


In [ ]:
margin = 0.5
enso.loc[enso['ENSO']>=margin,'phase'] = "C"
enso.loc[enso['ENSO']<=-margin,'phase'] = "W"
enso

In [ ]:
phase_color = {'W':'red', 'C':'blue'}
enso_selected = enso[(enso['decimal_year']>1925)&(enso['decimal_year']<2008)&(~enso['phase'].isna())]

#fig = go.Figure()
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=df_sorted['age_absolute'], y=df_sorted['d13c'],  mode="markers", marker_color="lightgreen",  name='d18O'),secondary_y=False)
fig.add_trace(go.Scatter(x=df_sorted['age_absolute'], y=df_sorted['d13c_ma'], mode="lines", line_color='green', name='average'), secondary_y=False)

fig.add_trace(go.Scatter(x=enso['decimal_year'], y=enso['ENSO'], mode="lines", line_color='gray', name='ENSO'), secondary_y=True)

fig.add_vline(x=1935.25, line_width=2, line_dash="dash", line_color="orange", opacity=1, showlegend=True,name='Gizo EQ 1935 Mar')
fig.add_vline(x=1966, line_width=2, line_dash="dash", line_color="violet", opacity=1, showlegend=True,name='Kavachi submarine volcano 1966')
fig.add_vline(x=1977.3, line_width=2, line_dash="dash", line_color="gold", opacity=1, showlegend=True,name='Guadalcanal 1977 EQ M9+')
fig.add_vline(x=2007, line_width=2, line_dash="dash", line_color="purple", opacity=1, showlegend=True,name='Gizo 2007 EQ M8.1')

for i, index in enumerate(enso_selected.index):
    
    df_iter = enso.loc[index]
    df_iter_next = enso.loc[index+1]
    x = df_iter['decimal_year']
    dx = df_iter_next['decimal_year'] - x
    fig.add_vrect(x0=x-dx, x1=dx+x, fillcolor=phase_color[df_iter['phase']],line_color=None, opacity=0.1)




# Add figure title
fig.update_layout(title_text="d13C and ENSO", template='simple_white')
# Set x-axis title
fig.update_xaxes(title_text="Year")

# Set y-axes titles
fig.update_yaxes(title_text= '$\delta C ^{13}$', secondary_y=False)
fig.update_yaxes(title_text="ENSO Index", secondary_y=True)

fig.show()
#fig.write_html('d13C_ENSO.html')
